In [1]:
import pandas as pd
import re
import numpy as np
from scipy import stats
import itertools
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import mean_squared_error
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.stats import pearsonr
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.colors import to_hex

In [2]:
# Load data UTHealth
uthh = pd.read_csv("C:/Users/jjm262/OneDrive - Yale University/Documents/Documents/00yale/04fourthyear/01projects/03tclock/02results/02trans_age-07272025/00uthhealth_rnaagecalc-07272025.csv")
# Rename the first column to SampleID
uthh.rename(columns={uthh.columns[0]: "SampleID"}, inplace=True)

# Load data GSE
gse = pd.read_csv("C:/Users/jjm262/OneDrive - Yale University/Documents/Documents/00yale/04fourthyear/01projects/03tclock/02results/02trans_age-07272025/02gse102556-08052025.csv")
# Rename the first column to SampleID
gse.rename(columns={gse.columns[0]: "SampleID"}, inplace=True)

# Load data VABB
vabb = pd.read_csv("C:/Users/jjm262/OneDrive - Yale University/Documents/Documents/00yale/04fourthyear/01projects/03tclock/02results/02trans_age-07272025/01vabb_rnaagecalc-07272025.csv")
# Rename the first column to SampleID
vabb.rename(columns={vabb.columns[0]: "SampleID"}, inplace=True)
# Replace the letter between numbers with an underscore
vabb["SampleID"] = vabb["SampleID"].str.replace(r"(?<=\d)[A-Z]+(?=\d)", "_", regex=True)

# Load data
dpclo = pd.read_csv("C:/Users/jjm262/OneDrive - Yale University/Documents/Documents/00yale/04fourthyear/01projects/03tclock/02results/90summaries/00deepclock-08032025.csv")
# Rename and keep only SampleID and KPANN_brain columns
dpclo = dpclo.rename(columns={'ID': 'SampleID', 'Predicted': 'KPANN_brain'})[['SampleID', 'Age', 'KPANN_brain', 'Bank']]
# Add GSE from GEO cohort
# Load data
gse_kpann = pd.read_csv("C:/Users/jjm262/OneDrive - Yale University/Documents/Documents/00yale/04fourthyear/01projects/03tclock/02results/90summaries/04gsecohort_kpnaa_clock_prediction-08052025.txt", sep = '\t')
gse_kpann = gse_kpann.rename(columns={'ID': 'SampleID', 'Predicted': 'KPANN_brain'})[['SampleID', 'Age', 'KPANN_brain', 'Bank']]
# Concatenate data
# Concatenate rows
kpann = pd.concat([dpclo, gse_kpann], axis=0, ignore_index=True)

# Merge with both filtered datasets
merged_df_com = pd.merge(kpann, uthh, on='SampleID')
merged_df2_com = pd.merge(kpann, vabb, on='SampleID')
merged_df3_com = pd.merge(kpann, gse, on='SampleID')
print(merged_df_com.shape)
print(merged_df2_com.shape)
print(merged_df3_com.shape)

(89, 212)
(546, 212)
(96, 219)


In [3]:
# --- 1. Subset relevant columns from both datasets ---
# Defining a function to extract the columns with brains and Age
def extract_age_and_brain(df, source_label):
    # Use 'AgeDeath' if available, otherwise fallback to 'Age'
    age_col = 'AgeDeath' if 'AgeDeath' in df.columns else 'Age'
    # Get columns that end with '_brain'
    brain_cols = [col for col in df.columns if col.endswith('_brain')]
    # Extract relevant columns
    subset = df[[age_col] + brain_cols].copy()
    # Rename age column to a common name
    subset.rename(columns={age_col: 'Age'}, inplace=True)
    # Drop rows with any NA values
    subset = subset.dropna()
    # Add source column
    subset['Source'] = source_label
    return subset

# Create subsets
uth_subset = extract_age_and_brain(merged_df_com, 'UTHealth')
vabb_subset = extract_age_and_brain(merged_df2_com, 'VABB')
gse_subset = extract_age_and_brain(merged_df3_com, 'GSE102556')
print(uth_subset.shape)
print(vabb_subset.shape)
print(gse_subset.shape)

# --- 2. Combine three datasets by row ---
# --- 1. Check column differences ---
cols_uth  = set(uth_subset.columns)
cols_vabb = set(vabb_subset.columns)
cols_gse  = set(gse_subset.columns)

print("Columns in UTH but not in VABB:", cols_uth - cols_vabb)
print("Columns in VABB but not in UTH:", cols_vabb - cols_uth)
print("Columns in UTH but not in GSE:", cols_uth - cols_gse)
print("Columns in GSE but not in UTH:", cols_gse - cols_uth)

# --- 2. Align all DataFrames to the same columns ---
# Union of all column names
all_cols = sorted(set(uth_subset.columns) | set(vabb_subset.columns) | set(gse_subset.columns))

# Reindex each DataFrame to have the same columns in the same order
uth_aligned  = uth_subset.reindex(columns=all_cols)
vabb_aligned = vabb_subset.reindex(columns=all_cols)
gse_aligned  = gse_subset.reindex(columns=all_cols)

# --- 3. Concatenate row-wise ---
combined_df = pd.concat([uth_aligned, vabb_aligned, gse_aligned], ignore_index=True)

(89, 11)
(546, 11)
(96, 11)
Columns in UTH but not in VABB: set()
Columns in VABB but not in UTH: set()
Columns in UTH but not in GSE: set()
Columns in GSE but not in UTH: set()


In [8]:
# --- Load your full dataset ---
df = combined_df

# --- Create Age² term ---
df["Age2"] = df["Age"] ** 2

# --- Identify clock columns ---
clock_cols = [col for col in df.columns if col.endswith("_brain")]

# --- Function to compute r, p-value, R², and N ---
def correlation_stats(x, y):
    # drop missing
    mask = x.notna() & y.notna()
    x, y = x[mask], y[mask]
    if len(x) < 3:  # too few values to compute
        return pd.Series({"r": None, "pval": None, "R2": None, "N": len(x)})
    r, p = pearsonr(x, y)
    return pd.Series({"r": r, "pval": p, "R2": r**2, "N": len(x)})

# --- Compute correlations overall ---
overall_results = []
for clock in clock_cols:
    stats = correlation_stats(df[clock], df["Age2"])
    stats["Clock"] = clock
    stats["Source"] = "All"
    overall_results.append(stats)

# --- Compute correlations per Source ---
per_source_results = []
for src, group in df.groupby("Source"):
    for clock in clock_cols:
        stats = correlation_stats(group[clock], group["Age2"])
        stats["Clock"] = clock
        stats["Source"] = src
        per_source_results.append(stats)

# --- Combine results ---
results = pd.concat([pd.DataFrame(overall_results),
                     pd.DataFrame(per_source_results)],
                     ignore_index=True)

# --- Reorder columns ---
results = results[["Clock", "Source", "r", "R2", "pval", "N"]]

# --- Save to CSV ---
results.to_csv("00SuplTable3-08272025.csv", index=False)

print("Saved correlations with r, R², p-values, and N to 00SuplTable3-08272025.csv")

Saved correlations with r, R², p-values, and N to 00SuplTable3-08272025.csv


In [9]:
# Read both files
ret = 'C:/Users/jjm262/OneDrive - Yale University/Documents/Documents/00yale/04fourthyear/01projects/03tclock/02results/06tranningclock-08152025/00RetrainedPredictions_gse102556-08172025.csv'
enRet = pd.read_csv(ret, index_col=0)
ret2 = 'C:/Users/jjm262/OneDrive - Yale University/Documents/Documents/00yale/04fourthyear/01projects/03tclock/02results/06tranningclock-08152025/00RetrainedPredictions_uthealth-08182025.csv'
enRet2 = pd.read_csv(ret2, index_col=0)

# Add source column to each
enRet["Source"] = "GSE102556"
enRet2["Source"] = "UTHealth"

# Concatenate along rows (so Source column is preserved)
combined_df2 = pd.concat([enRet, enRet2], axis=0, ignore_index=False)

# Reset index if you want a clean continuous index
combined_df2.reset_index(inplace=True)
combined_df2.rename(columns={"index": "SampleID"}, inplace=True)
combined_df2

,SampleID,Actual_Age,Predicted_ElasticNet,Predicted_DeepLearning,Predicted_ElasticNet_Stochastic,Predicted_DeepLearning_Stochastic,Source
0,S14.BA11,47,47.286596,46.053800,21.956768,22.953129,GSE102556
1,S17.BA11,41,40.909524,22.240500,26.519688,25.946888,GSE102556
2,S20.BA11,31,32.850170,27.220861,14.666239,16.698282,GSE102556
3,S23.BA11,19,23.948171,23.157152,19.274508,21.772709,GSE102556
4,S28.BA11,46,45.836763,37.918415,25.872570,24.990147,GSE102556
...,...,...,...,...,...,...,...
180,A67995,63,50.348354,41.703583,30.864792,31.065182,UTHealth
181,A67996,36,49.019974,29.035337,23.333821,21.391320,UTHealth
182,A67997,61,48.260497,27.981165,23.822408,22.541843,UTHealth
183,A67998,28,35.980747,36.136463,17.568902,17.257830,UTHealth


In [11]:
# --- Load your full dataset ---
df = combined_df2

# --- Create Age² term ---
df["Age2"] = df["Actual_Age"] ** 2

# --- Identify clock columns ---
clock_cols = ['Predicted_ElasticNet', 'Predicted_DeepLearning', 'Predicted_ElasticNet_Stochastic', 'Predicted_DeepLearning_Stochastic']

# --- Function to compute r, p-value, R², and N ---
def correlation_stats(x, y):
    # drop missing
    mask = x.notna() & y.notna()
    x, y = x[mask], y[mask]
    if len(x) < 3:  # too few values to compute
        return pd.Series({"r": None, "pval": None, "R2": None, "N": len(x)})
    r, p = pearsonr(x, y)
    return pd.Series({"r": r, "pval": p, "R2": r**2, "N": len(x)})

# --- Compute correlations overall ---
overall_results = []
for clock in clock_cols:
    stats = correlation_stats(df[clock], df["Age2"])
    stats["Clock"] = clock
    stats["Source"] = "All"
    overall_results.append(stats)

# --- Compute correlations per Source ---
per_source_results = []
for src, group in df.groupby("Source"):
    for clock in clock_cols:
        stats = correlation_stats(group[clock], group["Age2"])
        stats["Clock"] = clock
        stats["Source"] = src
        per_source_results.append(stats)

# --- Combine results ---
results = pd.concat([pd.DataFrame(overall_results),
                     pd.DataFrame(per_source_results)],
                     ignore_index=True)

# --- Reorder columns ---
results = results[["Clock", "Source", "r", "R2", "pval", "N"]]

# --- Save to CSV ---
results.to_csv("00SuplTable3a-08272025.csv", index=False)

print("Saved correlations with r, R², p-values, and N to 00SuplTable3a-08272025.csv")

Saved correlations with r, R², p-values, and N to 00SuplTable3-08272025.csv
